Import libraries:

In [1]:
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

Import cleaned data:

In [2]:
df = pd.read_csv('clinical_trials_cleaned.csv', parse_dates=['start_date', 'completion_date'])
for col in ['phase', 'sponsor_type']:
    df[col] = df[col].astype('category')
df.shape

(119484, 9)

Data descriptives:

In [3]:
print(df['duration_days'].describe())

count    119484.000000
mean       1012.048006
std         976.760415
min           1.000000
25%         334.000000
50%         730.000000
75%        1400.000000
max       23042.000000
Name: duration_days, dtype: float64


In [4]:
print(df['phase'].value_counts())

phase
PHASE2           31698
PHASE1           29742
PHASE3           24338
PHASE4           20259
PHASE1/PHASE2     7139
PHASE2/PHASE3     3694
EARLY_PHASE1      2614
Name: count, dtype: int64


In [5]:
print(df['sponsor_type'].value_counts())

sponsor_type
INDUSTRY     58235
OTHER        53532
NIH           3588
OTHER_GOV     1736
NETWORK       1311
FED            864
INDIV          184
UNKNOWN         34
Name: count, dtype: int64


Train-test split:

This data is based on time, so we will use a chronological split:

In [6]:
split_date = pd.Timestamp('2019-01-01') # chosen such that the training set is about 80% of the data
train = df[df['start_date'] < split_date]
test = df[df['start_date'] >= split_date]
print(train.shape, test.shape)

(93954, 9) (25530, 9)


Fit a baseline model:

In [8]:
feature_cols = ['phase', 'enrollment', 'sponsor_type', 'num_sites']
categorical_cols = ['phase', 'sponsor_type']

X_train = train[feature_cols].copy()
y_train = train['duration_days']
X_test = test[feature_cols].copy()
y_test = test['duration_days']

for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

model = lgb.LGBMRegressor(random_state=42)
model.fit(X_train, y_train, categorical_feature=categorical_cols)

preds = model.predict(X_test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000220 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 526
[LightGBM] [Info] Number of data points in the train set: 93954, number of used features: 4
[LightGBM] [Info] Start training from score 1119.285587


In [9]:
print("MAE:", mean_absolute_error(y_test, preds))
print("RMSE:", np.sqrt(mean_squared_error(y_test, preds)))

MAE: 493.1720807328965
RMSE: 631.0643213916284
